# Coursework 1 — try things here

This notebook only calls `cw1.py` and your lab files. It is not submitted and not marked. Cells marked *after Lab 4* / *after Lab 5* need those files.

In [1]:
%load_ext autoreload
%autoreload 2
import time
from lab2 import load, successors, goal_reached
from lab3 import bfs, dfs, ucs, plan_cost, action_cost
from cw1 import *

## Part A: three ways to check a plan

`apply_plan` replays a plan and returns the final state, or says exactly which step failed and why. `is_valid_plan` asks whether the goal holds at the end; `is_simple_plan` asks whether any state was visited twice. Part B's challenges are judged with exactly these three.

In [2]:
domain, p2, acts2 = load("domain.pddl", "p2.pddl")
plan = ucs(p2, acts2).plan
print("valid:", is_valid_plan(p2, acts2, plan), " simple:", is_simple_plan(p2.init, acts2, plan), " cost:", plan_cost(plan))
try:
    apply_plan(p2.init, acts2, ["pick-up-tool(dslr, lab)", "take-photo(cat, dslr, lab)", "pick-up-tool(knife, lab)"])
except ValueError as e:
    print(e)

valid: True  simple: True  cost: 12
step 2: pick-up-tool(knife, lab) needs [('hand-empty',)]


The optimal plan is valid and simple. The three-step plan fails at step 2 because the camera is still in the hand: that message is the one you will debug every plan with. A plan that walks out and straight back is valid for nothing and not simple: try it.

## Part A: heuristics, and whether to trust them *(after Lab 4)*

Three heuristics, two properties. A heuristic is **admissible** if it never overestimates the true cost to the goal, and **consistent** if it never drops by more than an action costs. A* is optimal with the first; A* with a closed set needs the second. Functions 5 and 6 search for a counterexample instead of taking anybody's word.

In [3]:
from lab4 import goal_count, h_add, greedy, astar
for h in (goal_count, h_max, h_add):
    a = admissibility_counterexample(h, p2, acts2)
    c = consistency_counterexample(h, p2, acts2)
    print(f"{h.__name__:11s} at s0: {h(p2.init, p2.goal, acts2):3}   admissible: {a is None}   consistent: {c is None}")

goal_count  at s0:   3   admissible: True   consistent: True
h_max       at s0:   4   admissible: True   consistent: True
h_add       at s0:  10   admissible: False   consistent: False


On p2, `goal_count` and `h_max` pass both tests and `h_add` fails both: it adds up the cost of goal facts that share work, so it overestimates. `goal_count` is not admissible in general either: one `put-down` can make three goal facts true at once. Build a problem where that happens and check it.

### What admissibility buys, and what it costs

In [4]:
d, p4, acts4 = load("domain.pddl", "p4.pddl")
print("UCS      ", ucs(p4, acts4))
print("A* h_max ", astar(p4, acts4, h_max))       # optimal, and slow
print("A* h_add ", astar(p4, acts4, h_add))       # fast; optimal here, but not guaranteed
print("greedy   ", greedy(p4, acts4, h_add))

UCS       12 actions, cost 16 | 75118 expanded, frontier up to 55483
A* h_max  12 actions, cost 16 | 9251 expanded, frontier up to 15490
A* h_add  12 actions, cost 16 | 16 expanded, frontier up to 57
greedy    12 actions, cost 16 | 14 expanded, frontier up to 49


Four plans of cost 16. Only the first two are *guaranteed* optimal, and the admissible one needed 9,000 expansions where the inadmissible one needed 16. That is the trade: guarantees are expensive.

## Big is not hard *(after Lab 4)*

`p10.pddl` has all ten items, 532 grounded actions and about 10$^{13}$ states. Its goal is eight facts.

In [5]:
d, p10, acts10 = load("domain.pddl", "p10.pddl")
t = time.time(); r = greedy(p10, acts10, goal_count); print("greedy, goal_count:", r, f"({time.time() - t:.2f}s)")
try:
    from lab5 import h_ff
    t = time.time(); r = greedy(p10, acts10, h_ff, 3_000); print("greedy, h_ff:      ", r, f"({time.time() - t:.0f}s)")
except ImportError:
    print("greedy, h_ff: after Lab 5")

greedy, goal_count: 21 actions, cost 35 | 144 expanded, frontier up to 512 (0.01s)
greedy, h_ff:       no plan | 3000 expanded, frontier up to 6503 (5s)


Greedy with the crudest heuristic solves a problem with 10$^{13}$ states in a hundredth of a second, with a plan of cost 35 that is in fact optimal (count the walks: three things must cross outdoors and the hand carries one). Size was never the difficulty.

The second line is the one to stare at: `h_ff`, the better heuristic everywhere else, finds **no plan** in 3,000 expansions. The goal contains a stack, and the delete relaxation never sees that a block stacked too early has to come off again, so greedy walks into a plateau and never leaves. Your `my_heuristic` is marked on three test problems, and one of them has a stack.

## Part B: your heuristic *(after Lab 4)*

We rank it by the states greedy search expands on three problems you have not seen, with a cap of 20,000 expansions each. Here it is on three you have, including the one with the stack.

In [ ]:
total = 0
for f in ["p4.pddl", "p6.pddl", "p10.pddl"]:
    d, p, acts = load("domain.pddl", f)
    t = time.time(); r = greedy(p, acts, my_heuristic, 20_000)
    ok = r.plan is not None and is_valid_plan(p, acts, r.plan)
    total += len(r.expanded)
    print(f"{f}: {r}  valid {ok}  ({time.time() - t:.1f}s)")
print("total expanded:", total)

## Part B: delivery *(after Lab 4)*

Four rooms, ten items each to be moved to another room, two photographs, one slicing, a hand that carries one thing. 

In [ ]:
d, pd, actsd = load("domain.pddl", "delivery.pddl")
print(len(actsd), "grounded actions;", len(pd.goal), "goal facts")
r = greedy(pd, actsd, goal_count, 20_000)
print("greedy, goal_count:", r, "| walks:", sum(a.startswith("walk") for a in r.plan))

In [ ]:
t = time.time(); plan = delivery_plan(30)
print(len(plan), "actions, cost", plan_cost(plan), "valid:", is_valid_plan(pd, actsd, plan), f"({time.time() - t:.0f}s)")
print("walks:", sum(a.startswith("walk") for a in plan))

## Part B: the tower

All ten items stacked in the lab in a given order, all photographed. Look at the goal: the nine `on-top` facts form one chain.

In [ ]:
d, pt, actst = load("domain.pddl", "tower.pddl")
print(sorted(f for f in pt.goal if f[0] == "on-top"))
t = time.time(); r = greedy(pt, actst, goal_count, 3_000); print("greedy, goal_count:", r, f"({time.time() - t:.0f}s)")

Greedy finds nothing here, and neither does any heuristic from the labs: every state in which some block sits on the wrong one looks almost finished to a heuristic that ignores deletes. The chain says which block must be placed first. The cell below runs your `tower_plan`.

In [ ]:
t = time.time(); plan = tower_plan(30)
print(len(plan), "actions, cost", plan_cost(plan), "valid:", is_valid_plan(pt, actst, plan), f"({time.time() - t:.0f}s)")

## Part B: the longest paths

A simple path never visits a state twice. `p2` has 2,560 states, so no simple plan has more than 2,559 actions; how close can you get? Both functions take a problem and its actions and must work on any world; they are ranked on `p2`.

In [ ]:
print("Lab 3's dfs, as a simple path:", len(dfs(p2, acts2, 3000).plan), "actions")
for fn in (longest_simple_plan, most_expensive_simple_plan):
    t = time.time(); plan = fn(p2, acts2, 20)
    print(f"{fn.__name__}: {len(plan)} actions, cost {plan_cost(plan)}, simple: {is_simple_plan(p2.init, acts2, plan)}  ({time.time() - t:.0f}s)")

Everything your searches did so far is the opposite of this problem: they looked for the *shortest* path and stopped early. Here the search has to keep going, avoid dead ends it cannot see yet, and back out when it is stuck. Which neighbour to try first matters enormously; so does knowing the whole 2,560-state graph before you start.

## Question 4: your own world

Needs `my_domain.pddl` and `my_problem.pddl` in this folder.

In [ ]:
d, mine, acts_mine = load("my_domain.pddl", "my_problem.pddl")
print(len(d.types), "types,", len(d.predicates), "predicates,", len(d.actions), "schemas,", len(mine.objects), "objects,", len(acts_mine), "grounded actions")
plan = my_plan()
print(len(plan), "actions, valid:", is_valid_plan(mine, acts_mine, plan))
print("shortest:", bfs(mine, acts_mine))